## Library Import

In [3]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import pandas as pd
import numpy as np

print("Imports OK")

Imports OK


In [4]:
# Load the strict de-duplicated set
strict = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_strict.csv")
# The 9 CAN features
feat = config.FEATURE_COLUMNS

# Separate STEERING signatures from benign signatures.
steering = strict[strict["true_class"] == "spoofing-STEERING_WHEEL"]
benign = strict[strict["true_class"] == "benign"]

print(f"STEERING signatures :   {len(steering)}")
print(f"benign signatures   :   {len(benign)}")

# Build a set of benign feature-tuples for fast lookup
benign_keys = set(map(tuple, benign[feat].itertuples(index=False, name=None)))

# For each STEERING signature check if its 9-feature vector exists in benign,
collisions = 0
for row in steering[feat].itertuples(index=False, name=None):
    if tuple(row) in benign_keys:
        collisions += 1
        print(f"    COLLISION:  {row} also exists as a benign frame")

print(f"\nSteering signatures that are byte-identical to a benign frame:    "
      f"{collisions} / {len(steering)}")

STEERING signatures :   3
benign signatures   :   3547

Steering signatures that are byte-identical to a benign frame:    0 / 3


In [5]:
# Load the RAW benign file directly (all 1.2M rows, before de-dup).
raw_benign = pd.read_csv(config.RAW_DIR / config.RAW_FILES["benign"])
raw_benign.columns = [c.strip() for c in raw_benign.columns]

# Build the set of ALL raw benign feature-tuples.
raw_benign_keys = set(map(tuple, raw_benign[feat].itertuples(index=False, name=None)))
print(f"Unique benign feature-vectors in RAW benign: {len(raw_benign_keys):,}")

# Re-check each STEERING signature against the full raw benign set.
collisions_raw = 0
for row in steering[feat].itertuples(index=False, name=None):
    if tuple(row) in raw_benign_keys:
        collisions_raw += 1
        print(f"  RAW COLLISION: {row} exists in raw benign traffic")

print(f"\nSTEERING signatures matching ANY raw benign frame: "
      f"{collisions_raw} / {len(steering)}")

Unique benign feature-vectors in RAW benign: 3,547

STEERING signatures matching ANY raw benign frame: 0 / 3
